In [11]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [2]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.gold')

DataFrame[]

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, DoubleType

# Silver feature tables (one row per package)
silver_github = spark.table('ddc_databricks.silver.github_package_features')
silver_pypi   = spark.table('ddc_databricks.silver.pypi_package_features')
silver_so     = spark.table('ddc_databricks.silver.so_package_features')

# Silver text tables for sentiment analysis
silver_so_questions  = spark.table('ddc_databricks.silver.so_questions')
silver_so_answers    = spark.table('ddc_databricks.silver.so_answers')
silver_readmes       = spark.table('ddc_databricks.silver.github_readmes')
silver_pypi_metadata = spark.table('ddc_databricks.silver.pypi_metadata')

In [4]:
# Sentiment analysis UDF, uses a pretrained HuggingFace transformer model as the primary
# analyser, falling back to a keyword heuristic only when the library is unavailable.

# Primary model: distilbert-base-uncased-finetuned-sst-2-english
#   A DistilBERT model fine-tuned on Stanford Sentiment Treebank v2 (SST-2).
#   Returns POSITIVE / NEGATIVE with a calibrated confidence score in [0.5, 1.0].
#   The confidence is mapped to compound score in [-1, +1] so results are comparable
#   to the rest of the pipeline:
#       compound = confidence   (POSITIVE)
#       compound = -confidence  (NEGATIVE)

sentiment_schema = StructType([
    StructField('compound', DoubleType(), True),
    StructField('positive', DoubleType(), True),
    StructField('negative', DoubleType(), True),
    StructField('neutral', DoubleType(), True),
])

# Characters sent to the tokeniser.  DistilBERT has a 512-token limit;
# 2000 chars = 500 tokens at roughly 4 chars/token
_SENTIMENT_MAX_CHARS = 2000

# Worker-process-level cache so the model is loaded only once per executor.
_hf_pipeline_cache = {}

def _get_hf_pipeline():
    """Load the HuggingFace sentiment pipeline once per worker process and cache it."""
    if 'pipe' not in _hf_pipeline_cache:
        try:
            from transformers import pipeline as hf_pipeline
            _hf_pipeline_cache['pipe'] = hf_pipeline(
                'sentiment-analysis',
                model='distilbert-base-uncased-finetuned-sst-2-english',
                truncation=True,
                max_length=512,
            )
        except Exception as e:
            print(f'HuggingFace pipeline unavailable: {e}')
            _hf_pipeline_cache['pipe'] = None
    return _hf_pipeline_cache['pipe']

In [5]:
def _compute_sentiment(text):
    if text is None or len(str(text).strip()) == 0:
        return (0.0, 0.0, 0.0, 1.0)

    snippet = str(text)[:_SENTIMENT_MAX_CHARS]

    # 1. Pretrained transformer model (DistilBERT fine-tuned on SST-2)
    pipe = _get_hf_pipeline()
    if pipe is not None:
        try:
            result  = pipe(snippet, truncation=True, max_length=512)[0]
            label   = result['label']   # 'POSITIVE' or 'NEGATIVE'
            score   = float(result['score'])  # confidence in [0.5, 1.0]
            if label == 'POSITIVE':
                pos, neg = score, 1.0 - score
            else:
                pos, neg = 1.0 - score, score
            compound = pos - neg  # maps to [-1, +1]
            return (compound, pos, neg, 0.0)
        except Exception:
            pass

    # 2. Keyword heuristic fallback
    lower = snippet.lower()
    _POS = ['good', 'great', 'excellent', 'easy', 'fast', 'useful', 'best',
            'perfect', 'works', 'love', 'helpful', 'nice', 'clean', 'awesome']
    _NEG = ['bad', 'error', 'bug', 'slow', 'broken', 'fail', 'crash',
            'wrong', 'issue', 'problem', 'terrible', 'awful', 'hate', 'worst']
    pos = sum(1 for w in _POS if w in lower)
    neg = sum(1 for w in _NEG if w in lower)
    total = pos + neg
    if total == 0:
        return (0.0, 0.0, 0.0, 1.0)
    compound = (pos - neg) / total
    return (float(compound), float(pos / total),
            float(neg / total), float(1.0 - abs(compound)))

sentiment_udf = F.udf(_compute_sentiment, sentiment_schema)
print('Sentiment UDF registered.')
print('  Primary:  distilbert-base-uncased-finetuned-sst-2-english (HuggingFace pretrained)')
print('  Fallback: keyword heuristic')

Sentiment UDF registered.
  Primary:  distilbert-base-uncased-finetuned-sst-2-english (HuggingFace pretrained)
  Fallback: keyword heuristic


In [6]:
# 1) Sentiment analysis on all text sources -> gold.package_sentiment

# Four sources with weights reflecting how directly they express developer opinion:
#   SO questions      30% developer questions about the package (broad signal)
#   SO answers        30% developer answers (detailed, often opinionated)
#   GitHub README     25% official project tone set by maintainers
#   PyPI description  15% marketing/summary copy (least opinionated)

so_q_sent = (
    silver_so_questions
    .select('package_name', 'question_id', 'question_body_text')
    .withColumn('s', sentiment_udf(F.col('question_body_text')))
    .groupBy('package_name')
    .agg(
        F.avg('s.compound').alias('so_question_sentiment_avg'),
        F.avg('s.positive').alias('so_question_sentiment_pos_avg'),
        F.avg('s.negative').alias('so_question_sentiment_neg_avg'),
        F.count('question_id').alias('so_question_sentiment_count'),
    )
)

so_a_sent = (
    silver_so_answers
    .select('package_name', 'answer_id', 'answer_body_text')
    .withColumn('s', sentiment_udf(F.col('answer_body_text')))
    .groupBy('package_name')
    .agg(
        F.avg('s.compound').alias('so_answer_sentiment_avg'),
        F.avg('s.positive').alias('so_answer_sentiment_pos_avg'),
        F.avg('s.negative').alias('so_answer_sentiment_neg_avg'),
        F.count('answer_id').alias('so_answer_sentiment_count'),
    )
)

readme_sent = (
    silver_readmes
    .select('package_name', 'readme_clean_text')
    .withColumn('s', sentiment_udf(F.col('readme_clean_text')))
    .select('package_name',
            F.col('s.compound').alias('readme_sentiment_compound'),
            F.col('s.positive').alias('readme_sentiment_pos'),
            F.col('s.negative').alias('readme_sentiment_neg'))
)

pypi_sent = (
    silver_pypi_metadata
    .select('package_name', 'description_clean_text')
    .withColumn('s', sentiment_udf(F.col('description_clean_text')))
    .select('package_name',
            F.col('s.compound').alias('pypi_desc_sentiment_compound'),
            F.col('s.positive').alias('pypi_desc_sentiment_pos'),
            F.col('s.negative').alias('pypi_desc_sentiment_neg'))
)

print('All four sentiment DataFrames built.')

All four sentiment DataFrames built.


In [7]:
# Weighted overall sentiment, four sources joined, missing sources default to 0.0.
# Weights sum to 1.0: SO questions 30%, SO answers 30%, README 25%, PyPI 15%.

package_sentiment = (
    so_q_sent.alias('q')
    .join(so_a_sent.alias('a'),   on='package_name', how='full')
    .join(readme_sent.alias('r'), on='package_name', how='full')
    .join(pypi_sent.alias('d'),   on='package_name', how='full')
    .withColumn(
        'overall_sentiment',
        F.coalesce(F.col('so_question_sentiment_avg'),      F.lit(0.0)) * 0.30
        + F.coalesce(F.col('so_answer_sentiment_avg'),      F.lit(0.0)) * 0.30
        + F.coalesce(F.col('readme_sentiment_compound'),    F.lit(0.0)) * 0.25
        + F.coalesce(F.col('pypi_desc_sentiment_compound'), F.lit(0.0)) * 0.15
    )
    .withColumn('scored_at', F.current_timestamp())
)

(
    package_sentiment
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.gold.package_sentiment')
)

package_sentiment.select(
    'package_name',
    'so_question_sentiment_avg',
    'so_answer_sentiment_avg',
    'readme_sentiment_compound',
    'pypi_desc_sentiment_compound',
    'overall_sentiment',
).orderBy('package_name').show(truncate=False)

+------------+-------------------------+-----------------------+-------------------------+----------------------------+---------------------+
|package_name|so_question_sentiment_avg|so_answer_sentiment_avg|readme_sentiment_compound|pypi_desc_sentiment_compound|overall_sentiment    |
+------------+-------------------------+-----------------------+-------------------------+----------------------------+---------------------+
|NULL        |NULL                     |NULL                   |NULL                     |0.0                         |0.0                  |
|django      |-0.11622222222222223     |-0.05366666666666666   |0.0                      |NULL                        |-0.05096666666666666 |
|fastapi     |0.24447619047619057      |0.28122222222222215    |0.3333333333333333       |NULL                        |0.24104285714285711  |
|flask       |-0.26481481481481484     |-0.17194444444444446   |0.3333333333333333       |NULL                        |-0.04769444444444444 |
|numpy

In [ ]:
# 2) Cross-source unified feature table → gold.package_unified_features
#    Joins GitHub, PyPI, and Stack Overflow feature tables with the full sentiment
#    breakdown per package — one wide row per package, ready for analytics and ML.

unified = (
    silver_github.alias('g')
    .join(silver_pypi.alias('p'), on='package_name', how='full')
    .join(silver_so.alias('s'),   on='package_name', how='full')
    .join(
        package_sentiment.select(
            'package_name',
            'so_question_sentiment_avg',
            'so_answer_sentiment_avg',
            'readme_sentiment_compound',
            'pypi_desc_sentiment_compound',
            'overall_sentiment',
        ).alias('sent'),
        on='package_name', how='left'
    )
    .withColumn('unified_at', F.current_timestamp())
)

(
    unified
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.gold.package_unified_features')
)

print(f'gold.package_unified_features: {unified.count()} packages')
unified.select(
    'package_name', 'stars', 'forks', 'last_month', 'questions_total', 'overall_sentiment'
).orderBy('package_name').show(truncate=False)


gold.package_unified_features: 9 packages
+------------+-----+-----+----------+---------------+---------------------+
|package_name|stars|forks|last_month|questions_total|overall_sentiment    |
+------------+-----+-----+----------+---------------+---------------------+
|NULL        |NULL |NULL |NULL      |NULL           |NULL                 |
|django      |87138|33795|NULL      |300            |-0.05096666666666666 |
|fastapi     |96644|8951 |NULL      |300            |0.24104285714285711  |
|flask       |71382|16760|NULL      |300            |-0.04769444444444444 |
|numpy       |31671|12205|NULL      |300            |-0.13262857142857143 |
|pandas      |48254|19789|NULL      |300            |0.0958               |
|requests    |53881|9799 |NULL      |300            |-0.11956666666666665 |
|scikit-learn|65543|26855|NULL      |300            |-0.3060833333333334  |
|torch       |98621|27320|NULL      |300            |-0.016811904761904764|
+------------+-----+-----+----------+---------

In [9]:
# 3) Compute health scores and save to gold.package_health_scores
#
# Each metric is log-transformed (where appropriate) and min-max scaled to [0,1]
# across the tracked packages, then combined into four weighted sub-scores:
#   GitHub Health   30%  — stars, forks, recency, contributors, docs
#   PyPI Adoption   25%  — downloads, momentum, releases, maturity
#   Community       25%  — SO volume, answer rate/quality, engagement
#   Sentiment       20%  — aggregate sentiment from all text sources

def _minmax(col_name, inverse=False):
    """Min-max scale a column across all rows to [0, 1]."""
    w = Window.partitionBy(F.lit(1))
    mn = F.min(F.col(col_name)).over(w)
    mx = F.max(F.col(col_name)).over(w)
    rng = mx - mn
    if inverse:
        return F.when(rng == 0, F.lit(0.5)).otherwise((mx - F.col(col_name)) / rng)
    return F.when(rng == 0, F.lit(0.5)).otherwise((F.col(col_name) - mn) / rng)

scoring = (
    unified.select(
        'package_name',
        # GitHub raw metrics
        F.log1p(F.coalesce(F.col('stars'), F.lit(0)).cast('double')).alias('log_stars'),
        F.log1p(F.coalesce(F.col('forks'), F.lit(0)).cast('double')).alias('log_forks'),
        F.coalesce(F.col('days_since_last_push'), F.lit(365)).cast('double').alias('days_since_push'),
        F.log1p(F.coalesce(F.col('contributors_total'), F.lit(0)).cast('double')).alias('log_contribs'),
        (F.lit(1.0) - F.coalesce(F.col('top_contributor_share'), F.lit(0.5))).alias('contrib_diversity'),
        F.log1p(F.coalesce(F.col('readme_word_count'), F.lit(0)).cast('double')).alias('log_readme_words'),
        F.when(F.coalesce(F.col('is_archived'), F.lit(False)), 0.0).otherwise(1.0).alias('not_archived'),
        F.when(F.coalesce(F.col('has_wiki'), F.lit(False)), 1.0).otherwise(0.0).alias('wiki_flag'),
        # PyPI raw metrics
        F.log1p(F.coalesce(F.col('last_month'), F.lit(0)).cast('double')).alias('log_monthly_dl'),
        F.coalesce(F.col('downloads_momentum_30d_vs_90d'), F.lit(0.33)).cast('double').alias('dl_momentum'),
        F.log1p(F.coalesce(F.col('release_count'), F.lit(0)).cast('double')).alias('log_releases'),
        F.coalesce(F.col('classifiers_count'), F.lit(0)).cast('double').alias('classifiers'),
        F.when(F.coalesce(F.col('has_license'), F.lit(False)), 1.0).otherwise(0.0).alias('license_flag'),
        F.when(F.coalesce(F.col('has_requires_python'), F.lit(False)), 1.0).otherwise(0.0).alias('req_py_flag'),
        # Community raw metrics
        F.log1p(F.coalesce(F.col('questions_total'), F.lit(0)).cast('double')).alias('log_questions'),
        F.coalesce(F.col('question_answered_rate'), F.lit(0.0)).cast('double').alias('answer_rate'),
        F.coalesce(F.col('avg_answer_score'), F.lit(0.0)).cast('double').alias('avg_ans_score'),
        F.log1p((F.coalesce(F.col('unique_question_askers'), F.lit(0))
                 + F.coalesce(F.col('unique_answerers'), F.lit(0))).cast('double')).alias('log_community'),
        F.coalesce(F.col('community_activity_30d'), F.lit(0)).cast('double').alias('activity_30d'),
        # Sentiment
        F.coalesce(F.col('overall_sentiment'), F.lit(0.0)).cast('double').alias('sentiment'),
    )
    # Normalize each metric to [0, 1]
    .withColumn('n_stars', _minmax('log_stars'))
    .withColumn('n_forks', _minmax('log_forks'))
    .withColumn('n_recency', _minmax('days_since_push', inverse=True))
    .withColumn('n_contribs', _minmax('log_contribs'))
    .withColumn('n_diversity', _minmax('contrib_diversity'))
    .withColumn('n_docs', _minmax('log_readme_words'))
    .withColumn('n_downloads', _minmax('log_monthly_dl'))
    .withColumn('n_momentum', _minmax('dl_momentum'))
    .withColumn('n_releases', _minmax('log_releases'))
    .withColumn('n_classifiers', _minmax('classifiers'))
    .withColumn('n_maturity',
                (F.col('license_flag') + F.col('req_py_flag') + F.col('n_classifiers')) / 3.0)
    .withColumn('n_so_volume', _minmax('log_questions'))
    .withColumn('n_answer_rate', _minmax('answer_rate'))
    .withColumn('n_ans_quality', _minmax('avg_ans_score'))
    .withColumn('n_community', _minmax('log_community'))
    .withColumn('n_activity', _minmax('activity_30d'))
    .withColumn('n_sentiment', (F.col('sentiment') + 1.0) / 2.0)
)

health_scores = (
    scoring
    .withColumn('github_score', F.round(
        (F.col('n_stars') * 0.25 + F.col('n_forks') * 0.10 + F.col('n_recency') * 0.20
         + F.col('n_contribs') * 0.15 + F.col('n_diversity') * 0.10
         + F.col('n_docs') * 0.10 + F.col('not_archived') * 0.05
         + F.col('wiki_flag') * 0.05) * 100, 2))
    .withColumn('pypi_score', F.round(
        (F.col('n_downloads') * 0.35 + F.col('n_momentum') * 0.20
         + F.col('n_releases') * 0.20 + F.col('n_maturity') * 0.25) * 100, 2))
    .withColumn('community_score', F.round(
        (F.col('n_so_volume') * 0.25 + F.col('n_answer_rate') * 0.20
         + F.col('n_ans_quality') * 0.20 + F.col('n_community') * 0.20
         + F.col('n_activity') * 0.15) * 100, 2))
    .withColumn('sentiment_score', F.round(F.col('n_sentiment') * 100, 2))
    .withColumn('overall_health_score', F.round(
        F.col('github_score') * 0.30 + F.col('pypi_score') * 0.25
        + F.col('community_score') * 0.25 + F.col('sentiment_score') * 0.20, 2))
    .withColumn('health_tier',
                F.when(F.col('overall_health_score') >= 80, 'A')
                .when(F.col('overall_health_score') >= 60, 'B')
                .when(F.col('overall_health_score') >= 40, 'C')
                .otherwise('D'))
    .withColumn('scored_at', F.current_timestamp())
    .select('package_name', 'github_score', 'pypi_score', 'community_score',
            'sentiment_score', 'overall_health_score', 'health_tier', 'scored_at')
)

(
    health_scores
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.gold.package_health_scores')
)

health_scores.orderBy(F.col('overall_health_score').desc()).show(truncate=False)

+------------+------------+----------+---------------+---------------+--------------------+-----------+--------------------------+
|package_name|github_score|pypi_score|community_score|sentiment_score|overall_health_score|health_tier|scored_at                 |
+------------+------------+----------+---------------+---------------+--------------------+-----------+--------------------------+
|pandas      |89.34       |45.83     |91.71          |54.79          |72.15               |B          |2026-03-27 16:22:40.794677|
|torch       |99.8        |45.83     |74.26          |49.16          |69.79               |B          |2026-03-27 16:22:40.794677|
|django      |90.55       |45.83     |84.83          |47.45          |69.32               |B          |2026-03-27 16:22:40.794677|
|numpy       |91.27       |45.83     |81.33          |43.37          |67.85               |B          |2026-03-27 16:22:40.794677|
|fastapi     |83.24       |45.83     |72.81          |62.05          |67.04        

In [10]:
spark.table('ddc_databricks.bronze.pypi_downloads_recent').select('_pypi_package', 'data').show(3, truncate=False)
spark.table('ddc_databricks.bronze.pypi_downloads_overall').count()

+-------------+------------------------------------------------------------+
|_pypi_package|data                                                        |
+-------------+------------------------------------------------------------+
|django       |{{1672519, 46951827, 12266550}, django, recent_downloads}   |
|fastapi      |{{18754910, 353517088, 90709812}, fastapi, recent_downloads}|
|flask        |{{7540464, 218474063, 48173614}, flask, recent_downloads}   |
+-------------+------------------------------------------------------------+
only showing top 3 rows


8